In [1]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")

In [17]:
from databricks_langchain import ChatDatabricks

# Using default authentication (e.g., from environment variables)
llm = ChatDatabricks(
    model="databricks-claude-3-7-sonnet",
    temperature=0,
    max_tokens=500,
    timeout=30.0,  # Timeout in seconds
    max_retries=3,  # Maximum number of retries
)

# Using a sdk.WorkspaceClient instance for custom authentication
from databricks.sdk import WorkspaceClient

#workspace_client = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
workspace_client = WorkspaceClient()
host = workspace_client.config.host

#llm = ChatOpenAI(model="gemma4:latest", temperature=0)

llm = ChatDatabricks(
    model="databricks-gemma-3-12b",
    workspace_client=workspace_client,
)

Invoke:

In [18]:
messages = [
    ("system", "You are a helpful translator. Translate the user sentence to French."),
    ("human", "I love programming."),
]
llm.invoke(messages)

AIMessage(content='Here are a few options for translating "I love programming" into French, with slight nuances:\n\n*   **J\'adore programmer.** (This is the most common and natural translation. "Adorer" means "to adore" or "to love a lot.")\n*   **J\'aime programmer.** (This is a simpler translation, using "aimer" which means "to like" or "to love." It\'s perfectly acceptable, just slightly less emphatic than "adorer.")\n\n\n\nI would recommend **J\'adore programmer.**', additional_kwargs={}, response_metadata={'usage': {'prompt_tokens': 27, 'completion_tokens': 112, 'total_tokens': 139}, 'prompt_tokens': 27, 'completion_tokens': 112, 'total_tokens': 139, 'model': 'gemma-3-12b-it-060225', 'model_name': 'gemma-3-12b-it-060225', 'finish_reason': 'stop'}, id='lc_run--019e7da4-305d-7da1-8187-99cefd2752d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 112, 'total_tokens': 139})

Stream:

In [19]:
for chunk in llm.stream(messages):
    print(chunk)

content='' additional_kwargs={} response_metadata={} id='lc_run--019e7da7-775f-79f3-9dc1-c002e8ce6c95' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 1, 'total_tokens': 28} tool_call_chunks=[]
content='Here' additional_kwargs={} response_metadata={} id='lc_run--019e7da7-775f-79f3-9dc1-c002e8ce6c95' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 1, 'total_tokens': 28} tool_call_chunks=[]
content=' are' additional_kwargs={} response_metadata={} id='lc_run--019e7da7-775f-79f3-9dc1-c002e8ce6c95' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 2, 'total_tokens': 29} tool_call_chunks=[]
content=' a' additional_kwargs={} response_metadata={} id='lc_run--019e7da7-775f-79f3-9dc1-c002e8ce6c95' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 3, 'total_tokens': 30} tool_call_chunks=[]
content=' few' additional_kwargs={} response_metad

In [ ]:
stream = llm.stream(messages)
full = next(stream)
for chunk in stream:
    full += chunk
full

To get token usage returned when streaming, pass the stream_usage kwarg:

In [ ]:
stream = llm.stream(messages, stream_usage=True)
next(stream).usage_metadata

Async:

In [ ]:
await llm.ainvoke(messages)

# stream:
# async for chunk in llm.astream(messages)

# batch:
# await llm.abatch([messages])

Tool calling:

In [ ]:
from pydantic import BaseModel, Field


class GetWeather(BaseModel):
    '''Get the current weather in a given location'''

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


class GetPopulation(BaseModel):
    '''Get the current population in a given location'''

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


llm_with_tools = llm.bind_tools([GetWeather, GetPopulation])
ai_msg = llm_with_tools.invoke(
    "Which city is hotter today and which is bigger: LA or NY?"
)
ai_msg.tool_calls

Custom inputs and outputs:

In [ ]:
# Pass custom inputs to the model
messages = [("human", "Hello, how are you?")]
custom_inputs = {"user_id": "12345", "session_id": "abc123"}

response = llm.invoke(messages, custom_inputs=custom_inputs)
print(response.content)

# Access custom outputs from the response (if provided by the model)
if hasattr(response, "custom_outputs"):
    print(f"Custom outputs: {response.custom_outputs}")

In [ ]:
# Custom inputs also work with streaming
for chunk in llm.stream(messages, custom_inputs=custom_inputs):
    print(chunk.content, end="")

    # Check for custom outputs in chunks
    if hasattr(chunk, "custom_outputs"):
        print(f"Custom outputs: {chunk.custom_outputs}")